# LAB04｜Final：ADK + Groq + Skill + MCP + LINE
## V2.1 學員教學版｜Google ADK + Groq + Agent Skills + MCP + LINE

> **適合對象：** 不熟 Python、第一次接觸 AI Agent 的學員  
> **操作方式：** 由上到下依序按 ▶ 執行。  
> **重要原則：** 看到「👨‍🎓 學生修改區」才需要動手，其餘程式請先不要修改。


# 🏁 Final Project

## 業務報價決策 Agent

完整流程：

```text
LINE
  ↓
Webhook
  ↓
Google ADK Agent
  ├─ Groq Model
  ├─ quotation-decision Skill
  └─ MCP Tools
        ↓
   客戶 / 產品 / 成本 / 歷史 / 產能
  ↓
APPROVE / REVIEW / REJECT
  ↓
LINE 回覆
```

### 這不是「聊天機器人」

它會：

1. 理解詢價內容
2. 載入公司 Skill
3. 透過 MCP 查企業資料
4. 做決策
5. 回報風險
6. 需要時要求人工確認


In [ ]:
# ============================================================
# STEP 1｜安裝 Final Project 套件
# ============================================================
#
# 除了 ADK / MCP / LiteLLM 之外，
# Final Project 還需要：
#
# flask    → 接收 LINE Webhook
# pyngrok  → 讓 Colab 暫時有公開 HTTPS 網址
# requests → 呼叫 LINE Reply API
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

!pip -q install "google-adk[mcp]>=2.0.0,<3.0.0" "litellm>=1.84,<2" pandas flask pyngrok requests
print("✅ Final Project 套件安裝完成")


In [ ]:
# ============================================================
# STEP 2｜上傳課程資料包
# ============================================================
# 這一格的目的：
#   把「企業模擬資料、Agent Skill、MCP Server」一次載入 Colab。
#
# 你要準備的檔案：
#   quotation_agent_v2_runtime.zip
#
# 👨‍🎓 學生修改區：
#   不用改程式，只要執行後選擇上述 ZIP 檔。
#
# ✅ 成功畫面：
#   看到「✅ 課程資料已載入」
#
# ❗常見錯誤：
#   - 上傳錯檔：重新執行這一格並選正確 ZIP。
#   - 沒看到 skills/ 或 data/：代表 ZIP 沒有成功解壓。
# ============================================================

from google.colab import files
import zipfile, os

if not os.path.exists("skills/quotation-decision/SKILL.md"):
    uploaded = files.upload()
    package = next((n for n in uploaded if n.endswith(".zip")), None)
    if not package:
        raise RuntimeError("❌ 請上傳 quotation_agent_v2_runtime.zip")
    with zipfile.ZipFile(package, "r") as z:
        z.extractall(".")

print("✅ 課程資料已載入")
print("目前重要資料夾：")
print(" - data/")
print(" - skills/quotation-decision/")
print(" - mcp_server/")


In [ ]:
# ============================================================
# STEP 3｜從 Colab Secrets 讀取 Groq API Key
# ============================================================
# 這一格的目的：
#   從 Google Colab 左側的「🔑 Secrets」安全讀取 GROQ_API_KEY。
#
# 👨‍🎓 學生事前準備：
#   1. 打開 Colab 左側選單
#   2. 點「🔑 Secrets」
#   3. 新增名稱：GROQ_API_KEY
#   4. Value 貼上你的 Groq API Key
#   5. 開啟「Notebook access」授權
#
# ⚠️ 不要把 API Key 直接寫在 Notebook 程式碼中。
#
# ✅ 成功畫面：
#   顯示「✅ 已從 Colab Secrets 讀取 GROQ_API_KEY」
#
# ❗常見錯誤：
#   KeyError / Secret not found
#   → 代表 Secrets 裡沒有 GROQ_API_KEY，或沒有開 Notebook access。
# ============================================================

import os
from google.colab import userdata

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")

    if not GROQ_API_KEY:
        raise ValueError("GROQ_API_KEY 為空值")

    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

    # 課堂穩定模型
    GROQ_MODEL = "groq/qwen/qwen3.8-27b"

    print("✅ 已從 Colab Secrets 讀取 GROQ_API_KEY")
    print("✅ Groq 設定完成")
    print("Model =", GROQ_MODEL)
    print("教學穩定模式：reasoning_effort = none")

except Exception as e:
    raise RuntimeError(
        "❌ 無法讀取 Colab Secret：GROQ_API_KEY\n\n"
        "請到 Colab 左側「🔑 Secrets」新增：\n"
        "名稱：GROQ_API_KEY\n"
        "值：你的 Groq API Key\n"
        "並開啟 Notebook access。\n\n"
        f"原始錯誤：{e}"
    )


## STEP 4｜同時載入 Skill 與 MCP

這一格是整堂課最關鍵的組合。


In [ ]:
# ============================================================
# STEP 4｜建立 SkillToolset + McpToolset
# ============================================================
#
# Skill：
#   告訴 Agent「報價應該怎麼判斷」
#
# MCP：
#   提供 Agent「客戶、產品、成本、歷史、產能」事實
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

from pathlib import Path
import sys

from google.adk.skills import load_skill_from_dir
from google.adk.tools import skill_toolset
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

# ① 載入企業 Skill
quotation_skill = load_skill_from_dir(
    Path("skills/quotation-decision")
)

skill_tools = skill_toolset.SkillToolset(
    skills=[quotation_skill]
)

# ② 連接 MCP Server
mcp_tools = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,
            args=["mcp_server/quotation_mcp_server.py"],
        )
    )
)

print("✅ Skill + MCP 已準備完成")


## STEP 5｜建立 Final Agent

請注意這一次 Agent 已經同時擁有：

- **Model**：Groq
- **Skill**：Quotation Decision
- **MCP**：企業資料 Tools


In [ ]:
# ============================================================
# STEP 5｜建立 Final Quotation Agent
# ============================================================
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
#
# 這一格執行成功，就代表：
#   Agent 架構已經完成。
# ============================================================

from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

root_agent = LlmAgent(
    name="quotation_decision_agent",

    model=LiteLlm(
        model=GROQ_MODEL,
        reasoning_effort="none",
    ),

    description="企業業務報價決策 Agent",

    instruction="""
你是製造業業務報價決策 Agent。

工作原則：
1. 報價問題必須使用 quotation-decision skill。
2. 必須使用 MCP tools 取得客戶、產品、成本、歷史成交與產能資料。
3. 不得自行創造企業資料。
4. 依 Skill 規則輸出 APPROVE / REVIEW / REJECT。
5. REVIEW 要指出需要哪個角色人工確認。
6. 使用繁體中文。
7. 輸出適合 LINE 閱讀。
8. 這是決策建議，不是正式對客戶的報價承諾。
""",

    tools=[
        skill_tools,
        mcp_tools
    ],
)

print("✅ Final Agent 建立完成")


In [ ]:
# ============================================================
# 共用工具｜讓我們可以直接「問 Agent 問題」
# ============================================================
# 這一格的目的：
#   Google ADK 的 Agent 需要 Runner + Session 才能真正執行。
#
# 你可以把它想成：
#
#   Agent = 員工
#   Runner = 讓員工開始工作的執行器
#   Session = 這一次對話的工作區
#
# 👨‍🎓 學生需要修改嗎？
#   不需要。
# ============================================================

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
import uuid

APP_NAME = "quotation_training"
USER_ID = "student"

async def ask_agent(agent, text: str):
    session_service = InMemorySessionService()
    session_id = str(uuid.uuid4())

    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )

    runner = Runner(
        agent=agent,
        app_name=APP_NAME,
        session_service=session_service,
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=text)]
    )

    final_answer = "(沒有取得最終回答)"

    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            texts = [
                p.text for p in event.content.parts
                if getattr(p, "text", None)
            ]
            if texts:
                final_answer = "\n".join(texts)

    return final_answer

print("✅ Agent 執行工具已準備完成")


## STEP 6｜先不要急著接 LINE

**一定先在 Colab 裡確認 Agent 本身會正常回答。**

👨‍🎓 學生修改區：只改 `QUESTION`。


In [ ]:
# ============================================================
# 👨‍🎓 學生修改區
# ============================================================

QUESTION = """
ABC Electronics 要 CV420H 2萬米，
30天交貨，需要105度耐熱、RoHS、客製印字，
這張可以報嗎？請給建議價格與風險。
"""

# ============================================================
# ⚠️ 以下不需要修改
# ============================================================

answer = await ask_agent(root_agent, QUESTION)

print("🤖 Agent 回覆：")
print(answer)


### ✅ 預期成功結果

回覆至少應該包含：

- 客戶
- 產品
- 數量
- 成本
- 歷史成交
- 產能
- APPROVE / REVIEW / REJECT
- 風險
- 下一步

**如果這一格都失敗，先不要接 LINE。**


## STEP 7｜準備 LINE 連線資訊

👨‍🎓 **學生只需要在執行後貼入 3 個值：**

1. LINE Channel Access Token
2. LINE Channel Secret
3. ngrok Authtoken


In [ ]:
# ============================================================
# 👨‍🎓 學生輸入區
# ============================================================

from getpass import getpass

LINE_CHANNEL_ACCESS_TOKEN = getpass(
    "請貼上 LINE Channel Access Token："
).strip()

LINE_CHANNEL_SECRET = getpass(
    "請貼上 LINE Channel Secret："
).strip()

NGROK_AUTHTOKEN = getpass(
    "請貼上 ngrok Authtoken："
).strip()

print("✅ LINE / ngrok 金鑰已載入")


## STEP 8｜建立 LINE 可以呼叫的 Agent

這段程式比較進階，**學生不用理解全部**。

只要記住：

```text
LINE 訊息
   ↓
run_quote_agent_sync()
   ↓
ADK Agent
   ↓
回傳文字
```


In [ ]:
# ============================================================
# STEP 8｜建立 LINE 可呼叫的 Agent 執行函式
# ============================================================
#
# ⚠️ 學生不需要修改。
#
# 為什麼這段比較長？
# 因為 Flask 是同步的，但 ADK Agent 執行是非同步的，
# 所以我們另外建立一個 asyncio event loop。
# ============================================================

import asyncio
import threading
import uuid

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

agent_loop = asyncio.new_event_loop()

def _run_loop():
    asyncio.set_event_loop(agent_loop)
    agent_loop.run_forever()

threading.Thread(
    target=_run_loop,
    daemon=True
).start()

async def run_quote_agent_async(text: str):
    session_service = InMemorySessionService()
    session_id = str(uuid.uuid4())

    user_id = "line-user"
    app_name = "quotation_line_app"

    await session_service.create_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id,
    )

    runner = Runner(
        agent=root_agent,
        app_name=app_name,
        session_service=session_service,
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=text)]
    )

    final_text = "目前沒有取得 Agent 最終回覆。"

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=message,
    ):
        if event.is_final_response() and event.content and event.content.parts:
            texts = [
                p.text for p in event.content.parts
                if getattr(p, "text", None)
            ]
            if texts:
                final_text = "\n".join(texts)

    return final_text

def run_quote_agent_sync(text: str):
    future = asyncio.run_coroutine_threadsafe(
        run_quote_agent_async(text),
        agent_loop,
    )
    return future.result(timeout=120)

print("✅ LINE Agent 執行層建立完成")


## STEP 9｜啟動 LINE Webhook

這一格執行後會產生一個網址，例如：

```text
https://xxxx.ngrok-free.app/webhook
```

等一下要把它貼到 LINE Developers。


In [ ]:
# ============================================================
# STEP 9｜啟動 Flask + LINE Webhook + ngrok
# ============================================================
#
# 功能：
# 1. LINE 把訊息 POST 到 /webhook
# 2. 驗證 x-line-signature
# 3. 把文字交給 Agent
# 4. 使用 LINE Reply API 回覆
#
# ⚠️ 學生不需要修改。
# ============================================================

from flask import Flask, request, abort
from pyngrok import ngrok

import requests
import hmac
import hashlib
import base64
import time

app = Flask(__name__)

def verify_line_signature(
    raw_body: bytes,
    signature: str
) -> bool:

    digest = hmac.new(
        LINE_CHANNEL_SECRET.encode("utf-8"),
        raw_body,
        hashlib.sha256,
    ).digest()

    expected = base64.b64encode(
        digest
    ).decode("utf-8")

    return hmac.compare_digest(
        expected,
        signature or ""
    )

def line_reply(
    reply_token: str,
    text: str
):
    url = "https://api.line.me/v2/bot/message/reply"

    headers = {
        "Authorization": f"Bearer {LINE_CHANNEL_ACCESS_TOKEN}",
        "Content-Type": "application/json",
    }

    payload = {
        "replyToken": reply_token,
        "messages": [
            {
                "type": "text",
                "text": text[:4900]
            }
        ],
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=30
    )

    print(
        "LINE Reply:",
        response.status_code,
        response.text[:300]
    )

@app.route("/webhook", methods=["POST"])
def webhook():
    raw = request.get_data()

    signature = request.headers.get(
        "x-line-signature",
        ""
    )

    if not verify_line_signature(
        raw,
        signature
    ):
        abort(400)

    body = request.get_json(
        silent=True
    ) or {}

    for event in body.get("events", []):

        is_text_message = (
            event.get("type") == "message"
            and event.get("message", {}).get("type") == "text"
        )

        if is_text_message:
            text = event["message"]["text"]
            reply_token = event.get("replyToken")

            print("收到 LINE：", text)

            answer = run_quote_agent_sync(
                text
            )

            if reply_token:
                line_reply(
                    reply_token,
                    answer
                )

    return "OK", 200

ngrok.set_auth_token(
    NGROK_AUTHTOKEN
)

public_url = ngrok.connect(
    5000,
    "http"
).public_url

threading.Thread(
    target=lambda: app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    ),
    daemon=True
).start()

time.sleep(1)

print("✅ Webhook 啟動成功")
print("🌐 請複製以下網址到 LINE Developers：")
print(public_url + "/webhook")


## STEP 10｜LINE Developers 設定

把上一格產生的：

```text
https://xxxxx.ngrok-free.app/webhook
```

貼到 LINE Developers：

1. 進入 Messaging API Channel
2. 找到 **Webhook URL**
3. 貼上網址
4. 按 **Verify**
5. 開啟 **Use webhook**
6. 加入 LINE Official Account 為好友

### 手機測試

傳送：

> ABC Electronics 要 CV420H 2萬米，30天交貨，需要105度耐熱、RoHS、客製印字，這張可以報嗎？

如果成功，你會直接收到 Agent 的決策結果。


## ❗LAB04 常見錯誤排除

### LINE Verify 失敗
先確認：

- Colab 這格還在執行
- ngrok URL 沒有換
- URL 最後有 `/webhook`
- Channel Secret 正確

### LINE 收到訊息但沒有回覆
檢查：

- Channel Access Token
- Colab Secret：GROQ_API_KEY
- Colab Console 是否有紅色錯誤

### `401 Unauthorized`
通常是 Groq 或 LINE Token 錯誤。

### MCP Server 啟動錯誤
確認：

```text
mcp_server/quotation_mcp_server.py
```

是否存在。

### ngrok 顯示 authentication failed
請重新確認 ngrok Authtoken。


# 🎓 Final Project 驗收

請逐項確認：

- [ ] Google ADK Agent 能執行
- [ ] 模型使用 Groq
- [ ] Agent 能載入 quotation-decision Skill
- [ ] Agent 能透過 MCP 查企業資料
- [ ] Agent 能判斷 APPROVE / REVIEW / REJECT
- [ ] REVIEW 會要求人工確認
- [ ] LINE 能發送問題
- [ ] LINE 能收到報價決策結果

## 你已經完成的企業 Agent 架構

```text
LLM       → Groq
Agent     → Google ADK
Skill     → Agent Skills
Tools     → MCP
Interface → LINE
Human     → REVIEW 人工核准
```

### 最重要的一句話

> **LLM 負責理解，Skill 負責教工作方法，MCP 負責拿企業事實，Agent 負責完成流程，Human 負責高風險決策。**
